In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
import glob,math,re,gc
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
n_gpus = torch.cuda.device_count()
print(f"device: {device}, GPUs available: {n_gpus}")
if device == "cuda":
    print(f"GPU 0: {torch.cuda.get_device_name(0)}")

def show_gpu_mem(tag=""):
    if device == "cuda":
        for i in range(n_gpus):
            alloc = torch.cuda.memory_allocated(i) / 1e9
            reserved = torch.cuda.memory_reserved(i) / 1e9
            print(f"[{tag}] GPU {i}: {alloc:.2f} GB allocated, {reserved:.2f} GB reserved")

def free_memory():
    gc.collect()
    torch.cuda.empty_cache()

show_gpu_mem("startup")

device: cuda, GPUs available: 2
GPU 0: Tesla T4
[startup] GPU 0: 0.00 GB allocated, 0.00 GB reserved
[startup] GPU 1: 0.00 GB allocated, 0.00 GB reserved


In [2]:
def load_data():
    path = glob.glob("/kaggle/input/**/*.txt",recursive=True)
    data_path = Path(path[0])
    with open(data_path,mode="r") as f:
        data = f.read()
        # Keep enough data for a meaningful validation split while retaining a
        # manageable subset for a notebook run. Set to 1.0 for the full corpus.
        data_fraction = 0.01
        length = data_fraction * len(data)
        data = data[:int(length)]
        print(f"Length of dataset in characters: {len(data)}")
    split = (0.8 * len(data))
    train_data,test_set = data[:int(split)],data[int(split):]
    # 80/10/10 train/validation/test split.
    valid_size = int(0.5 * len(test_set))
    valid_data, test_data = test_set[:valid_size], test_set[valid_size:]
    return train_data,test_data,valid_data

train_data,test_data,valid_data = load_data()
print(len(train_data),len(test_data),len(valid_data))

Length of dataset in characters: 1493263
1194610 149327 149326


### chunking

In [3]:
def chunk_data(data, chunk_size=240):
    # Preserve punctuation, numbers, and non-English text for the tokenizer.
    tokens = data.split()
    chunks = [" ".join(tokens[i:i + chunk_size]) for i in range(0, len(tokens), chunk_size)]
    return chunks

train_chunk = chunk_data(train_data)
test_chunk = chunk_data(test_data)
valid_chunk = chunk_data(valid_data)
print(len(train_chunk),len(test_chunk),len(valid_chunk))
print(train_chunk[0])
for i in range(3):
    print(f"Train chunk {i+1}: {train_chunk[i][:100]}...")
    print(f"Test chunk {i+1}: {test_chunk[i][:100]}...")
    print(f"Valid chunk {i+1}: {valid_chunk[i][:100]}...")

899 114 109
MARCH # All Stories New and Complete Publisher Editor IF is published bi-monthly by Quinn Publishing Company, Inc., Kingston, New York. Volume #, No. #. Copyright # by Quinn Publishing Company, Inc. Application for Entry' as Second Class matter at Post Office, Buffalo, New York, pending. Subscription # for # issues in U.S. and Possessions: Canada # for # issues; elsewhere #. Aiiow four weeks for change of address. All stories appearing in this magazine are fiction. Any similarity to actual persons is coincidental. #c a fcopy. Printed ia U.S. A. A chat with the editor i # science fiction magazine called IF. The title was selected after much thought because of its brevity and on the theory it is indicative of the field and will be easy to remember. The tentative title that just morning and couldn't remember it until we'd had a cup of coffee, it was summarily discarded. A great deal of thought and effort lias gone into the formation of this magazine. We have had the aid of sev

## tokenization

### we tokenize batches per trainig loop to avoid crashing

In [4]:
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

In [5]:
def tokenize_data(data,tokenizer,max_length=256):
    all_input_ids,all_mask_attention = [],[]
    for chunk in data:
        encoding = tokenizer(chunk,
                             truncation=True,
                             padding="max_length",
                             max_length=max_length,
                             return_tensors = "pt",
                             )
        all_input_ids.append(encoding["input_ids"])
        all_mask_attention.append(encoding["attention_mask"])
    return {"input_ids": torch.cat(all_input_ids,dim = 0), "attention_mask": torch.cat(all_mask_attention,dim = 0)}

probed_data = tokenize_data(test_chunk,tokenizer)
length = probed_data["input_ids"].sum(dim = 1)
print(f"Length of probed data: {length.max().item()}")
print(f"mean: {length.float().mean()}, var: {length.float().var()}")
print(probed_data["input_ids"].shape,probed_data["attention_mask"].shape)
print(probed_data["input_ids"][0],probed_data["attention_mask"][0])

Length of probed data: 6422597
mean: 4785061.5, var: 575535382528.0
torch.Size([114, 256]) torch.Size([114, 256])
tensor([     0,   1690,    111,   6863,  25188,   3688,      5,  47331,      7,
         64337,    717,  81206,     38,    276,    522, 133800,    177,  21533,
         25188,     83,  73432,   1257,    142, 167969,   3525,  44457,      4,
            10,  44457,     23,   3129,  29458,  12924,      7,    621,    308,
          3632,    297,  66161,     47,  60813,     10,  34475,  60042,      4,
          5045,     23,     70, 144996,     23,   3129,  23335,   1294,      4,
         24286,   4778,    765,   2809, 133888,      4,   1284, 129927,    214,
            23,    450,    242,     70,  41207,    271,      4,    229,  18557,
         12924,      7,    621,   5036,  11814,      4,    678,     70,   2684,
        225073,  53088,      5,  73831,   2271,   6664,      4,  74574,  53208,
             4,  20467,  23811,    555,    316,      4,  60971,  53208,      4,
      

In [6]:
encoded_train_data = tokenize_data(train_chunk,tokenizer)
encoded_test_data = tokenize_data(test_chunk,tokenizer)
encoded_valid_data = tokenize_data(valid_chunk,tokenizer)

encoded_train_data = {k:v.cpu() for k,v in encoded_train_data.items()}
encoded_test_data = {k:v.cpu() for k,v in encoded_test_data.items()}
encoded_valid_data = {k:v.cpu() for k,v in encoded_valid_data.items()}
print(encoded_train_data["input_ids"].shape,encoded_test_data["input_ids"].shape,encoded_valid_data["input_ids"].shape)

torch.Size([899, 256]) torch.Size([114, 256]) torch.Size([109, 256])


#### positional encoding for a model to know what a sequence or sentence means.

In [7]:
class positional_encoding(nn.Module):
    def __init__(self,embedded_dim, max_length = 512,dropout = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_length,embedded_dim)
        position = torch.arange(0,max_length).unsqueeze(1)
        div_term = torch.exp(torch.arange(0,embedded_dim,2)*(-math.log(10000.0)/embedded_dim))
        pe[:,0::2] = torch.sin(position*div_term)
        pe[:,1::2] = torch.cos(position*div_term)
        self.register_buffer('pe',pe.unsqueeze(0))

    def forward(self,x):
        x = x+self.pe[:,:x.size(1)]
        return self.dropout(x) 

show_gpu_mem("after positional encoding")       

[after positional encoding] GPU 0: 0.00 GB allocated, 0.00 GB reserved
[after positional encoding] GPU 1: 0.00 GB allocated, 0.00 GB reserved


# transformer

# building a multi-head attention
`q,k,v` -->query,key,value , which are `W(Q,K,V)` ,here W is weights

In [8]:
class Multihead_attention(nn.Module):
    def __init__(self,embedded_dim,nheads,dropout =0.3):
        super(Multihead_attention,self).__init__()

        self.embedded_dim = embedded_dim
        self.nheads = nheads
        self.head_dim = embedded_dim//nheads
# linear layer for q,k,v
        self.w_q = nn.Linear(embedded_dim,embedded_dim)
        self.w_k = nn.Linear(embedded_dim,embedded_dim)
        self.w_v = nn.Linear(embedded_dim,embedded_dim)
#output projection
        self.w_o = nn.Linear(embedded_dim,embedded_dim)
        self.dropout = nn.Dropout(dropout)
        self.scale = math.sqrt(self.head_dim)

    def forward(self,query,key,value,mask =None):
        Q = self.w_q(query)
        K = self.w_k(key)
        V = self.w_v(value)   

        #split inti multiple heads 
        Q= Q.view(Q.size(0),Q.size(1),self.nheads,self.head_dim).transpose(1,2)
        K = K.view(K.size(0),K.size(1),self.nheads,self.head_dim).transpose(1,2)
        V = V.view(V.size(0),V.size(1),self.nheads,self.head_dim).transpose(1,2)

        scores = torch.matmul(Q,K.transpose(-2,-1))/self.scale
        if mask is not None:
            if mask.dtype == torch.bool:
                scores = scores.masked_fill(~mask, float('-inf'))
            else:
                scores = scores.masked_fill(mask == 0, float('-inf'))
        attention_weights = torch.softmax(scores,dim=-1)
        attention_output = torch.matmul(self.dropout(attention_weights),V)

        attention_output = attention_output.transpose(1,2).contiguous().view(attention_output.size(0), -1, self.embedded_dim)
        output = self.w_o(attention_output)
        return output,attention_weights

In [9]:
attention_mask = encoded_train_data['attention_mask']
print(attention_mask.shape)

torch.Size([899, 256])


In [10]:
mha = Multihead_attention(embedded_dim= 512,nheads=8)
batch_size = 2
input_ids = encoded_train_data['input_ids'][:batch_size]
embeddings = nn.Embedding(tokenizer.vocab_size,512)
positional_enc = positional_encoding(embedded_dim=512)
x = embeddings(input_ids)
x = positional_enc(x)
query = key = value = x
mask = attention_mask[:batch_size, None, None, :] 
output,attention_weights = mha(query,key,value,mask = mask)
print(f"Output shape: {output.shape}, Attention weights shape: {attention_weights.shape}")
show_gpu_mem("after multihead attention")

Output shape: torch.Size([2, 256, 512]), Attention weights shape: torch.Size([2, 8, 256, 256])
[after multihead attention] GPU 0: 0.00 GB allocated, 0.00 GB reserved
[after multihead attention] GPU 1: 0.00 GB allocated, 0.00 GB reserved


## building a encoder which contains:
`multi-head attention`,`add-norm layer`,`feed forward network`,`add-norm layer`

In [11]:
class transformer_encoder(nn.Module):
    def __init__(self,input_dim,hidden_dim,n_heads,dropout):
        super(transformer_encoder,self).__init__()
        self.multihead_attention = Multihead_attention(embedded_dim=input_dim,nheads=n_heads,dropout=dropout)
        self.norm1 = nn.LayerNorm(input_dim)
        self.dropout1 = nn.Dropout(dropout)

        self.ff = nn.Sequential(
            nn.Linear(input_dim,hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim,input_dim),
        )
        self.norm2 = nn.LayerNorm(input_dim)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self,x,mask = None):
        attention_output,_ = self.multihead_attention(x,x,x,mask)
        x = self.norm1(x + self.dropout1(attention_output))
        ff_output = self.ff(x)
        x = self.norm2(x +self.dropout2(ff_output))   
        return x 

## building a decoder:
`self-attention`,`cross attention`,`ff`

In [12]:
class transformer_decoder(nn.Module):
    def __init__(self,input_dim,hidden_dim,n_heads,dropout):
        super(transformer_decoder,self).__init__()
        self.multihead_attention = Multihead_attention(input_dim,n_heads,dropout)
        self.norm1 = nn.LayerNorm(input_dim)
        self.dropout1 = nn.Dropout(dropout)
        self.cross_attention = Multihead_attention(input_dim,n_heads,dropout)
        self.norm2 = nn.LayerNorm(input_dim)
        self.dropout2 = nn.Dropout(dropout)

        self.ff = nn.Sequential(
            nn.Linear(input_dim,hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim,input_dim),
        )
        self.norm3 = nn.LayerNorm(input_dim)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self,x,encoder_output,causal_mask = None,cross_mask =None):
        attention_output,_ = self.multihead_attention(x,x,x,causal_mask)
        x = self.norm1(x+self.dropout1(attention_output))
        cross_output,_ = self.cross_attention(x,encoder_output,encoder_output,cross_mask)
        x = self.norm2(x+self.dropout2(cross_output))
        ff_out = self.ff(x)
        x = self.norm3(x+self.dropout3(ff_out))
        return x


In [13]:
def causal_mask(seq_length,device):
    mask = torch.tril(torch.ones(seq_length,seq_length,device=device)).bool()
    return mask[None,None,:,:]
seq_length = x.size(1)
causal_masks = causal_mask(seq_length,device=device)

# transformer architecture

In [14]:
class Transformer(nn.Module):
    def __init__(self, vocab_size, embedded_dim=256, n_heads=8, hidden_dim=512,
                 n_encoder_layers=2, n_decoder_layers=2, max_length=256, dropout=0.1):
        super().__init__()
        # The source and target are the same language in this next-word task.
        # Sharing embeddings substantially reduces VRAM use.
        self.target_embedding = nn.Embedding(vocab_size, embedded_dim)
        self.src_embedding = self.target_embedding
        self.positional_encode = positional_encoding(embedded_dim, max_length, dropout=dropout)

        self.encoder_block = nn.ModuleList([transformer_encoder(embedded_dim,hidden_dim,n_heads,dropout) for _ in range(n_encoder_layers)])
        self.decoder_block = nn.ModuleList([transformer_decoder(embedded_dim,hidden_dim,n_heads,dropout) for _ in range(n_decoder_layers)])

        self.output_projection = nn.Linear(embedded_dim, vocab_size, bias=False)
        self.output_projection.weight = self.target_embedding.weight

    def encode(self, src_ids, encoder_mask=None):
        x = self.src_embedding(src_ids)
        x = self.positional_encode(x)
        for layer in self.encoder_block:
            x = layer(x, encoder_mask)
        return x

    def decode(self, target_ids, encode_output, decoder_mask=None, cross_mask=None):
        x = self.target_embedding(target_ids)
        x = self.positional_encode(x)
        seq_length = x.size(1)
        causal = causal_mask(seq_length, x.device)
        if decoder_mask is not None:
            causal = causal & decoder_mask.bool()
        for layer in self.decoder_block:
            x = layer(x, encode_output, causal_mask=causal, cross_mask=cross_mask)
        return x

    def forward(self, src_ids, target_ids, encoder_mask=None, decoder_mask=None, cross_mask=None):
        # Masks are deliberately separate: encoder self-attention, decoder
        # self-attention, and cross-attention have different query/key shapes.
        if cross_mask is None:
            cross_mask = encoder_mask
        encode_output = self.encode(src_ids, encoder_mask)
        decode_output = self.decode(target_ids, encode_output, decoder_mask, cross_mask)
        logits = self.output_projection(decode_output)
        return logits

In [15]:
model = Transformer(vocab_size=tokenizer.vocab_size).to(device)
src_ids = encoded_train_data["input_ids"][:batch_size].to(device)
src_mask = attention_mask[:batch_size, None, None, :].to(device)
target_ids = src_ids.to(device)
logits = model(src_ids,target_ids,src_mask)
print(logits.shape)

torch.Size([2, 256, 250002])


In [16]:
# 1. Create a tiny test batch and push to device
src_test = encoded_train_data["input_ids"][:2].to(device)
tgt_test = src_test.clone()

# 2. Run forward pass
model.zero_grad()
logits_test = model(src_test, tgt_test)

# 3. Calculate a mock loss and backpropagate
loss_test = logits_test.sum()
loss_test.backward()

# 4. Check if gradients exist and are non-zero
embedding_grad = model.src_embedding.weight.grad
print("Are weights updating?", embedding_grad is not None and embedding_grad.sum().item() != 0)
print("Max absolute gradient value:", embedding_grad.abs().max().item() if embedding_grad is not None else "None")


Are weights updating? True
Max absolute gradient value: 65480.1328125


In [17]:
show_gpu_mem()

[] GPU 0: 1.67 GB allocated, 2.30 GB reserved
[] GPU 1: 0.00 GB allocated, 0.00 GB reserved


# building a training loop with gpu and memory optimization.
This training setup targets low-VRAM GPUs using a few standard techniques:
- **DataLoader + small `batch_size`** instead of loading the whole dataset onto the GPU at once
- **Mixed precision (`torch.cuda.amp`)** — runs most ops in fp16, roughly halving activation memory and speeding up matmuls
- **Gradient accumulation** — simulates a larger effective batch size without holding a large batch in memory
- **Gradient clipping** — keeps training stable, especially important with fp16
- **OOM catch-and-skip** — if a batch still doesn't fit, clear the cache and skip it rather than crashing the whole run
- **Explicit `del` + `torch.cuda.empty_cache()`** after each step/epoch to release memory PyTorch would otherwise hold in its caching allocator

In [18]:
import time
from torch.utils.data import TensorDataset,DataLoader
from torch.amp import autocast,GradScaler

def data_loader(encode_data, batch_size=2, shuffle=True):
    dataset = TensorDataset(encode_data["input_ids"], encode_data["attention_mask"])
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, pin_memory=True)

In [19]:
def train_one_epoch(model, data_loader, optimizer, scaler, criterion, device, accumulation_steps=8, max_grad_norm=1.0):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    total_loss = 0.0
    n_batches = 0
    accumulated_batches = 0
    amp_enabled = device == "cuda"

    def optimizer_step(batch_count):
        scaler.unscale_(optimizer)
        # Correct the final, partial accumulation window.
        if batch_count < accumulation_steps:
            for parameter in model.parameters():
                if parameter.grad is not None:
                    parameter.grad.mul_(accumulation_steps / batch_count)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

    for step, (input_ids, attention_mask) in enumerate(data_loader):
        input_ids = input_ids.to(device, non_blocking=True)
        attention_mask = attention_mask.to(device, non_blocking=True)
        decoder_input = input_ids[:, :-1]
        labels = input_ids[:, 1:]
        token_mask = attention_mask[:, :-1].bool()
        # Causal masks prevent the encoder/cross-attention path from seeing
        # the next token that is being predicted.
        sequence_mask = causal_mask(decoder_input.size(1), device) & token_mask[:, None, None, :]
        logits = batch_loss = loss = None

        try:
            with autocast(device_type="cuda", dtype=torch.float16, enabled=amp_enabled):
                logits = model(decoder_input, decoder_input, sequence_mask, sequence_mask, sequence_mask)
                batch_loss = criterion(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))
                loss = batch_loss / accumulation_steps
            scaler.scale(loss).backward()
            accumulated_batches += 1
            if accumulated_batches == accumulation_steps:
                optimizer_step(accumulated_batches)
                accumulated_batches = 0
            total_loss += batch_loss.item()
            n_batches += 1
        except torch.cuda.OutOfMemoryError:
            print(f"  [OOM] skipping batch {step}, clearing cache")
            optimizer.zero_grad(set_to_none=True)
            accumulated_batches = 0
            free_memory()
        finally:
            del input_ids, attention_mask, decoder_input, labels, token_mask, sequence_mask, logits, batch_loss, loss

    if accumulated_batches:
        optimizer_step(accumulated_batches)
    return total_loss / max(n_batches, 1)

In [20]:
@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    n_batches = 0
    
    amp_enabled = device == "cuda"
    for input_ids, attn_mask in dataloader:
        input_ids = input_ids.to(device, non_blocking=True)
        attn_mask = attn_mask.to(device, non_blocking=True)
        decoder_input = input_ids[:, :-1]
        labels = input_ids[:, 1:]
        token_mask = attn_mask[:, :-1].bool()
        sequence_mask = causal_mask(decoder_input.size(1), device) & token_mask[:, None, None, :]
        logits = loss = None
    
        try:
            with autocast(device_type="cuda", dtype=torch.float16, enabled=amp_enabled):
                logits = model(decoder_input, decoder_input, sequence_mask, sequence_mask, sequence_mask)
                loss = criterion(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))
            total_loss += loss.item()
            n_batches += 1
        except torch.cuda.OutOfMemoryError:
            print("  [OOM] skipping eval batch, clearing cache")
            torch.cuda.empty_cache()
            gc.collect()
            continue
    
        del input_ids, attn_mask, decoder_input, labels, token_mask, sequence_mask, logits, loss
    
    torch.cuda.empty_cache()
    return total_loss / max(n_batches, 1)


In [21]:
def find_latest_checkpoint(save_dir="checkpoints"):
    ckpt_dir = Path(save_dir)
    if not ckpt_dir.exists():
        return None
    checkpoints = list(ckpt_dir.glob("checkpoint_epoch*.pt"))
    if not checkpoints:
        return None
    checkpoints.sort(key=lambda p: int(p.stem.replace("checkpoint_epoch", "")))
    return checkpoints[-1]

def train(model, train_loader, val_loader, epochs=3, lr=2e-4, accumulation_steps=8, device=device,
          save_dir="checkpoints", save_every=1, resume_from=None):
    Path(save_dir).mkdir(parents=True, exist_ok=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr * 0.05)
    criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
    scaler = GradScaler(device=device, enabled=device == "cuda")

    start_epoch = 0
    prev_checkpoint_path = None

    if resume_from is not None:
        resume_from = Path(resume_from)
        checkpoint = torch.load(resume_from, map_location=device)

        model_to_load = model.module if isinstance(model, nn.DataParallel) else model
        model_to_load.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        scaler.load_state_dict(checkpoint["scaler_state_dict"])
        if "scheduler_state_dict" in checkpoint:
            scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

        start_epoch = checkpoint["epoch"]
        prev_checkpoint_path = resume_from
        print(f"resumed from {resume_from} — continuing at epoch {start_epoch + 1}/{epochs}")

    
    for epoch  in range(start_epoch, epochs):
        start = time.time()
        train_loss = train_one_epoch(model,train_loader,optimizer,scaler,criterion,device,accumulation_steps)
        val_loss = evaluate(model,val_loader,criterion,device)
        scheduler.step()
        total_time = time.time() - start
        print(f"epoch {epoch+1}/{epochs} | training loss : {train_loss:.4f} | val_loss : {val_loss:.4f} | time : {total_time:.2f}s")

        if (epoch + 1) % save_every == 0:
            checkpoint_path = Path(save_dir)/ f"checkpoint_epoch{epoch+1}.pt"
            model_to_save = model.module if isinstance(model, nn.DataParallel) else model
            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model_to_save.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scaler_state_dict": scaler.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "train_loss": train_loss,
                "val_loss": val_loss,
                }, checkpoint_path)
            print(f"Checkpoint saved to {checkpoint_path}")

            if prev_checkpoint_path is not None and prev_checkpoint_path.exists():
                            prev_checkpoint_path.unlink()
                            print(f"Deleted previous checkpoint {prev_checkpoint_path}")
            
            prev_checkpoint_path = checkpoint_path
        free_memory()  

    return model    

### training model

In [22]:
batch_size = 2
accumulation_steps = 8
save_every = 1
# New architecture: do not resume incompatible old 512-dimension checkpoints.
save_dir = "checkpoints_small"

In [ ]:
train_loader = data_loader(encoded_train_data,batch_size=batch_size,shuffle=True)
val_loader = data_loader(encoded_valid_data,batch_size=batch_size,shuffle=False)

model = Transformer(vocab_size=tokenizer.vocab_size, embedded_dim=256, hidden_dim=512,
                    n_encoder_layers=2, n_decoder_layers=2, max_length=256).to(device)
if n_gpus > 1:
    model = nn.DataParallel(model)
    print(f"using {n_gpus} GPUs via DataParallel")
resume_from = find_latest_checkpoint(save_dir) 
model = train(model, train_loader, val_loader, epochs=30, lr=2e-4, accumulation_steps=accumulation_steps,
              device=device, save_dir=save_dir, save_every=save_every, resume_from=resume_from)

using 2 GPUs via DataParallel
epoch 1/30 | training loss : 58.3003 | val_loss : 36.2555 | time : 58.82s
Checkpoint saved to checkpoints_small/checkpoint_epoch1.pt
epoch 2/30 | training loss : 34.8357 | val_loss : 29.9643 | time : 57.91s
Checkpoint saved to checkpoints_small/checkpoint_epoch2.pt
Deleted previous checkpoint checkpoints_small/checkpoint_epoch1.pt
epoch 3/30 | training loss : 28.9483 | val_loss : 25.4863 | time : 57.92s
Checkpoint saved to checkpoints_small/checkpoint_epoch3.pt
Deleted previous checkpoint checkpoints_small/checkpoint_epoch2.pt
epoch 4/30 | training loss : 24.4510 | val_loss : 21.5323 | time : 57.93s
Checkpoint saved to checkpoints_small/checkpoint_epoch4.pt
Deleted previous checkpoint checkpoints_small/checkpoint_epoch3.pt
epoch 5/30 | training loss : 20.8404 | val_loss : 18.5030 | time : 57.91s
Checkpoint saved to checkpoints_small/checkpoint_epoch5.pt
Deleted previous checkpoint checkpoints_small/checkpoint_epoch4.pt
epoch 6/30 | training loss : 18.1541 

In [ ]:
def _unwrap_model(model):
    return model.module if isinstance(model, nn.DataParallel) else model

def _full_mask(seq_len, device):
    return torch.ones((1, 1, seq_len, seq_len), dtype=torch.bool, device=device)

def _encode_prompt(prompt, device):
    encoding = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256,
        add_special_tokens=True,
    )
    return encoding["input_ids"].to(device)

@torch.no_grad()
def predict_next_token_logits(model, input_ids, device=device):
    base_model = _unwrap_model(model)
    base_model.eval()
    if input_ids.dim() == 1:
        input_ids = input_ids.unsqueeze(0)
    input_ids = input_ids.to(device)
    seq_len = input_ids.size(1)
    mask = _full_mask(seq_len, device)
    amp_enabled = device == "cuda"
    with autocast(device_type="cuda", dtype=torch.float16, enabled=amp_enabled):
        logits = base_model(input_ids, input_ids, mask, mask, mask)
    return logits[:, -1, :]

@torch.no_grad()
def generate_multinomial(model, prompt, max_new_tokens=40, temperature=1.0, top_k=None, device=device):
    base_model = _unwrap_model(model)
    base_model.eval()
    generated = _encode_prompt(prompt, device)
    eos_token_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else tokenizer.sep_token_id
    amp_enabled = device == "cuda"

    for _ in range(max_new_tokens):
        seq_len = generated.size(1)
        mask = _full_mask(seq_len, device)
        with autocast(device_type="cuda", dtype=torch.float16, enabled=amp_enabled):
            logits = base_model(generated, generated, mask, mask, mask)
            next_logits = logits[:, -1, :] / max(temperature, 1e-6)

        if top_k is not None and top_k > 0:
            top_k = min(top_k, next_logits.size(-1))
            values, indices = torch.topk(next_logits, top_k, dim=-1)
            probs = torch.softmax(values, dim=-1)
            sampled_index = torch.multinomial(probs[0], 1).item()
            next_token = indices[0, sampled_index].view(1, 1)
        else:
            probs = torch.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, 1)

        generated = torch.cat([generated, next_token], dim=1)
        if eos_token_id is not None and next_token.item() == eos_token_id:
            break

    return tokenizer.decode(generated[0], skip_special_tokens=True)

def _beam_score(score, seq_len, length_penalty):
    if length_penalty == 0:
        return score
    return score / (seq_len ** length_penalty)

@torch.no_grad()
def generate_beam_search(model, prompt, max_new_tokens=40, beam_size=4, length_penalty=0.7, device=device):
    base_model = _unwrap_model(model)
    base_model.eval()
    start_tokens = _encode_prompt(prompt, device)
    eos_token_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else tokenizer.sep_token_id
    amp_enabled = device == "cuda"
    beams = [(start_tokens, 0.0)]
    finished = []

    for _ in range(max_new_tokens):
        candidates = []

        for seq, score in beams:
            if eos_token_id is not None and seq[0, -1].item() == eos_token_id:
                finished.append((seq, score))
                continue

            seq_len = seq.size(1)
            mask = _full_mask(seq_len, device)
            with autocast(device_type="cuda", dtype=torch.float16, enabled=amp_enabled):
                logits = base_model(seq, seq, mask, mask, mask)
                log_probs = torch.log_softmax(logits[:, -1, :], dim=-1)

            top_scores, top_tokens = torch.topk(log_probs, beam_size, dim=-1)
            for token_score, token_id in zip(top_scores[0], top_tokens[0]):
                new_seq = torch.cat([seq, token_id.view(1, 1)], dim=1)
                candidates.append((new_seq, score + token_score.item()))

        if not candidates:
            break

        candidates.sort(key=lambda item: _beam_score(item[1], item[0].size(1), length_penalty), reverse=True)
        beams = candidates[:beam_size]

        if eos_token_id is not None and all(seq[0, -1].item() == eos_token_id for seq, _ in beams):
            finished.extend(beams)
            break

    pool = finished + beams
    best_seq, _ = max(pool, key=lambda item: _beam_score(item[1], item[0].size(1), length_penalty))
    return tokenizer.decode(best_seq[0], skip_special_tokens=True)

# Example usage:
# prompt = "The spaceship drifted quietly"
# print(generate_beam_search(model, prompt, max_new_tokens=30, beam_size=4))
# print(generate_multinomial(model, prompt, max_new_tokens=30, temperature=0.9, top_k=50))